# Анализ негативных отзывов Steam по extraction-шутерам

В этом проекте я собираю отзывы игроков из Steam по четырём extraction-шутерам, но дальше отдельно анализирую только негативные отзывы. Мне важно понять не общий рейтинг игры, а конкретные боли игроков: читы, оптимизация, серверы, баланс, монетизация, прогрессия и другие проблемы, которые нельзя повторять при проектировании своей игры.

**План работы:**
1. Импортирую библиотеки.
2. Задаю настройки: игры, языки, лимит отзывов.
3. Скачиваю или беру из кэша отзывы Steam.
4. Очищаю данные и выделяю негативные отзывы.
5. Считаю общие показатели жалоб по каждой игре.
6. Расширяю список горячих слов по категориям проблем.
7. Смотрю каждую игру отдельно: какие жалобы встречаются чаще всего.
8. Строю общую тепловую карту жалоб по играм и категориям.
9. Строю графики по датам: сколько негативных отзывов появляется во времени и какие темы всплывают чаще.
10. Собираю короткий итог для выводов по жанру.

Данные беру из публичного Steam Reviews API — это тот же источник, что показывает отзывы на странице магазина.

## Шаг 1. Импорт библиотек

Здесь всё стандартное:
- `urllib.request` — чтобы обращаться к Steam API (внешние пакеты не нужны).
- `json` — ответ Steam приходит в формате JSON.
- `time` — небольшие паузы между запросами, чтобы не долбить сервер.
- `re` — регулярные выражения для поиска слов в тексте отзывов.
- `pathlib.Path` — удобная работа с путями к папкам.
- `pandas` — таблицы, группировки, сохранение в CSV.
- `matplotlib` — графики.

In [ ]:
import http.client
import json
import re
import time
import urllib.request
import urllib.error
import urllib.parse
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

print("Библиотеки загружены")

## Шаг 2. Настройки

Задаю список игр с их App ID и параметры сбора.

- `GAMES` — какие игры анализируем.
- `LANGUAGES` — беру русские и английские негативные отзывы (два отдельных прохода).
- `MAX_NEGATIVE_REVIEWS_PER_GAME = None` — значит беру максимум негативных отзывов за всё время, пока Steam API не вернёт пустую страницу.
- `FORCE_REFRESH = True` в ячейке выгрузки нужен для полной перезагрузки данных, если хочу заново пройти весь Steam API.
- Папки для результатов создаются автоматически. Код заранее находит папку проекта, чтобы тетрадку можно было запускать и из корня репозитория, и из папки проекта.

In [ ]:
GAMES = {
    "Escape from Tarkov": 3932890,
    "Arena Breakout: Infinite": 2073620,
    "ARC Raiders": 1808500,
    "Delta Force": 2507950,
}

LANGUAGES = ["russian", "english"]

MAX_NEGATIVE_REVIEWS_PER_GAME = None

current_dir = Path.cwd()
if current_dir.name == "steam-reviews-analysis":
    project_dir = current_dir
elif (current_dir / "steam-reviews-analysis").exists():
    project_dir = current_dir / "steam-reviews-analysis"
else:
    project_dir = current_dir

output_dir = project_dir / "output"
raw_dir = output_dir / "raw"
raw_negative_dir = output_dir / "raw_negative"
charts_dir = output_dir / "charts"

for folder in (output_dir, raw_dir, raw_negative_dir, charts_dir):
    folder.mkdir(parents=True, exist_ok=True)

print("Папка проекта:", project_dir)
print("Игр в анализе:", len(GAMES))
print("Языки:", LANGUAGES)
print("Лимит негативных отзывов на игру:", MAX_NEGATIVE_REVIEWS_PER_GAME)

## Шаг 3. Функция скачивания негативных отзывов

Steam отдаёт отзывы постранично. За один запрос можно получить максимум 100 отзывов, а чтобы перейти на следующую страницу, нужно передать `cursor` из предыдущего ответа.

Здесь важно: я сразу передаю в запрос `review_type=negative`. Это значит, что Steam отдаёт только отзывы «Не рекомендую», а положительные отзывы вообще не скачиваются.

Логика функции:
1. Первый запрос идёт с `cursor="*"`.
2. Беру `filter=recent`, чтобы идти от свежих негативных отзывов к старым.
3. После каждого запроса складываю негативные отзывы в общий список и запоминаю новый `cursor`.
4. Если лимит `None`, иду до конца: пока Steam не вернёт пустую страницу или пока `cursor` не перестанет меняться.
5. При `RemoteDisconnected` и похожих сетевых ошибках делаю несколько повторов.
6. Между запросами делаю паузу, чтобы не перегружать сервер.

Из каждого негативного отзыва оставляю текст, дату, язык, наигранное время и число лайков отзыва.

In [ ]:
def get_negative_reviews(app_id, language, max_reviews=None):
    """Скачивает только негативные отзывы одной игры на одном языке."""
    collected = []
    cursor = "*"
    seen_cursors = set()

    while max_reviews is None or len(collected) < max_reviews:
        params = {
            "json": "1",
            "filter": "recent",
            "review_type": "negative",
            "language": language,
            "num_per_page": "100",
            "purchase_type": "all",
            "cursor": cursor,
        }
        url = "https://store.steampowered.com/appreviews/{}?{}".format(
            app_id, urllib.parse.urlencode(params)
        )
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

        data = None
        for attempt in range(1, 5):
            try:
                with urllib.request.urlopen(request, timeout=30) as response:
                    data = json.loads(response.read().decode("utf-8", errors="ignore"))
                break
            except (urllib.error.URLError, TimeoutError, http.client.RemoteDisconnected) as error:
                print(f"  Ошибка запроса, попытка {attempt}/4:", error)
                time.sleep(3 * attempt)

        if data is None:
            print("  Steam не ответил после 4 попыток — останавливаюсь")
            break

        if data.get("success") != 1:
            break

        reviews = data.get("reviews", [])
        if not reviews:
            break

        for r in reviews:
            author = r.get("author", {})
            collected.append({
                "recommendationid": r.get("recommendationid"),
                "language": r.get("language"),
                "review": (r.get("review") or "").replace("\n", " ").strip(),
                "voted_up": r.get("voted_up"),
                "votes_up": r.get("votes_up", 0),
                "timestamp_created": r.get("timestamp_created"),
                "playtime_at_review": author.get("playtime_at_review", 0),
                "num_games_owned": author.get("num_games_owned", 0),
            })

        new_cursor = data.get("cursor")
        if not new_cursor or new_cursor in seen_cursors:
            break
        seen_cursors.add(new_cursor)
        cursor = new_cursor

        if len(collected) % 5000 == 0:
            print(f"  уже скачано {len(collected)} негативных отзывов")
        time.sleep(0.35)

    if max_reviews is not None:
        collected = collected[:max_reviews]
    return collected


print("Функция get_negative_reviews готова")

## Шаг 4. Скачиваю негативные отзывы за всё время

Прохожу по каждой игре и по каждому языку, но запрашиваю у Steam только `review_type=negative`.

Если `MAX_NEGATIVE_REVIEWS_PER_GAME = None`, лимита нет: тетрадка идёт по страницам Steam API до конца доступной истории. Сырые негативные отзывы сохраняются отдельно в `output/raw_negative/`, чтобы не смешивать их со старыми файлами всех отзывов.

Если нужно заново перекачать данные, ставлю `FORCE_REFRESH = True`. После полной выгрузки можно вернуть `False`, чтобы не ждать повторной загрузки каждый запуск.

In [ ]:
FORCE_REFRESH = False

per_language_limit = None
if MAX_NEGATIVE_REVIEWS_PER_GAME is not None:
    per_language_limit = MAX_NEGATIVE_REVIEWS_PER_GAME // len(LANGUAGES)

all_frames = []

for game_name, app_id in GAMES.items():
    safe_name = re.sub(r"[^a-z0-9]+", "_", game_name.lower()).strip("_")
    raw_path = raw_negative_dir / f"{safe_name}_negative.csv"

    if raw_path.exists() and not FORCE_REFRESH:
        game_df = pd.read_csv(raw_path)
        print(f"{game_name}: взято из файла ({len(game_df)} негативных отзывов)")
    else:
        rows = []
        for language in LANGUAGES:
            reviews = get_negative_reviews(app_id, language, per_language_limit)
            print(f"{game_name} [{language}]: скачано {len(reviews)} негативных отзывов")
            rows.extend(reviews)
        game_df = pd.DataFrame(rows)
        game_df.insert(0, "game", game_name)
        game_df.insert(1, "app_id", app_id)
        game_df.to_csv(raw_path, index=False, encoding="utf-8")
        print(f"{game_name}: сохранено в {raw_path.name} ({len(game_df)} негативных отзывов)")

    if "game" not in game_df.columns:
        game_df.insert(0, "game", game_name)
    all_frames.append(game_df)

print("\nГотово. Собрано игр:", len(all_frames))

## Шаг 5. Собираю всё в одну таблицу

Склеиваю отзывы всех игр в один DataFrame и смотрю, сколько отзывов получилось по каждой игре и языку.

In [ ]:
df = pd.concat(all_frames, ignore_index=True)

print("Всего отзывов:", len(df))
print("\nОтзывов по играм:")
print(df["game"].value_counts())
print("\nОтзывов по языкам:")
print(df["language"].value_counts())

df.head()

## Шаг 6. Очистка и преобразование

Привожу данные к удобному виду:
- убираю дубликаты по `recommendationid` и пустые отзывы;
- перевожу `timestamp_created` (unix-время) в нормальную дату;
- перевожу наигранное время из минут в часы;
- добавляю столбец `sentiment` со значениями «Похвала» / «Жалоба» — так удобнее читать графики;
- добавляю `text_lower` — текст в нижнем регистре, по нему потом ищу ключевые слова.

Очищенную таблицу сохраняю в `output/clean_reviews.csv`.

In [ ]:
df = df.drop_duplicates(subset="recommendationid")
df = df[df["review"].astype(str).str.strip() != ""]

df["date"] = pd.to_datetime(df["timestamp_created"], unit="s", errors="coerce")
df["playtime_hours"] = (df["playtime_at_review"].fillna(0) / 60).round(1)
df["sentiment"] = df["voted_up"].map({True: "Похвала", False: "Жалоба"})
df["text_lower"] = df["review"].astype(str).str.lower()
df["review_length"] = df["review"].astype(str).str.len()

df = df.reset_index(drop=True)

clean_path = output_dir / "clean_reviews.csv"
df.to_csv(clean_path, index=False, encoding="utf-8")

print("После очистки осталось отзывов:", len(df))
print("Сохранено в:", clean_path.name)
df[["game", "language", "sentiment", "playtime_hours", "date", "review"]].head()

## Шаг 7. Общая картина по негативным отзывам

На этом этапе в таблице уже должны быть только негативные отзывы (`review_type=negative` в запросе Steam API). Положительные отзывы в проект больше не попадают.

Считаю общие показатели по каждой игре:
- сколько негативных отзывов удалось собрать за всё время;
- сколько из них на русском и английском;
- период, который покрывают отзывы;
- средняя наигранность авторов негативных отзывов;
- средняя длина негативного комментария.

Это базовая таблица для понимания масштаба жалоб по каждой игре.

In [ ]:
negative_reviews = df.copy()
negative_reviews = negative_reviews[negative_reviews["voted_up"] == False].copy()
negative_reviews["date_day"] = negative_reviews["date"].dt.date
negative_reviews["month"] = negative_reviews["date"].dt.to_period("M").astype(str)

negative_stats = negative_reviews.groupby("game").agg(
    негативных_отзывов=("recommendationid", "count"),
    первая_дата=("date", "min"),
    последняя_дата=("date", "max"),
    средняя_наигранность_ч=("playtime_hours", "mean"),
    средняя_длина_отзыва=("review_length", "mean"),
).reset_index()

language_stats = negative_reviews.pivot_table(
    index="game",
    columns="language",
    values="review",
    aggfunc="count",
    fill_value=0,
).reset_index()

complaint_summary = negative_stats.merge(language_stats, on="game", how="left")
complaint_summary["средняя_наигранность_ч"] = complaint_summary["средняя_наигранность_ч"].round(1)
complaint_summary["средняя_длина_отзыва"] = complaint_summary["средняя_длина_отзыва"].round(0).astype(int)
complaint_summary["первая_дата"] = complaint_summary["первая_дата"].dt.date
complaint_summary["последняя_дата"] = complaint_summary["последняя_дата"].dt.date

for language in LANGUAGES:
    if language not in complaint_summary.columns:
        complaint_summary[language] = 0

complaint_summary = complaint_summary[[
    "game",
    "негативных_отзывов",
    "russian",
    "english",
    "первая_дата",
    "последняя_дата",
    "средняя_наигранность_ч",
    "средняя_длина_отзыва",
]].sort_values("негативных_отзывов", ascending=False).reset_index(drop=True)

complaint_summary.to_csv(output_dir / "negative_complaint_summary.csv", index=False, encoding="utf-8")
negative_reviews.to_csv(output_dir / "negative_reviews.csv", index=False, encoding="utf-8")

print("Всего негативных отзывов:", len(negative_reviews))
complaint_summary

## Шаг 8. Расширенный список горячих слов по жалобам

Теперь размечаю негативные отзывы по категориям проблем. Список слов расширен: для каждой категории добавлены русские и английские варианты, сленг, частые формы слов и типичные формулировки из отзывов.

Категории сделаны под extraction-shooter:
- читы и нечестная игра;
- античит и баны;
- оптимизация, FPS, вылеты;
- серверы, пинг, десинхрон;
- баланс оружия, брони и матчей;
- монетизация и pay-to-win;
- прогрессия, гринд, вайпы, лут;
- стрельба, движение, TTK;
- баги и поломанные механики;
- сложность, порог входа и онбординг;
- контент, эндгейм и повторяемость;
- интерфейс, UX и лаунчер;
- звук, визуал и читаемость;
- комьюнити и токсичность;
- поддержка, разработчики и коммуникация.

Один отзыв может попасть сразу в несколько категорий.

In [ ]:
THEMES = {
    "Читы и нечестная игра": [
        "cheat", "cheater", "cheaters", "hacker", "hackers", "hacking", "aimbot", "wallhack", "wall hack",
        "esp", "radar hack", "speedhack", "speed hack", "soft aim", "no recoil", "macro", "scripts",
        "rage hacker", "blatant cheater", "cheaters everywhere", "full of cheaters",
        "чит", "читы", "читер", "читеры", "читаков", "читерство", "читерский", "хак", "хаки",
        "аимбот", "аим", "вх", "валхак", "волхак", "есп", "радар", "макрос", "софт", "без отдачи",
    ],
    "Античит и баны": [
        "anti-cheat", "anticheat", "anti cheat", "battleye", "eac", "easy anti cheat", "vac",
        "false ban", "false banned", "banned", "ban wave", "unban", "hwid ban", "shadow ban",
        "античит", "анти чит", "батлай", "battleye", "еас", "изичит", "ложный бан", "забанили",
        "забанен", "бан", "баны", "разбан", "теневой бан", "банит", "волна банов",
    ],
    "Оптимизация, FPS и вылеты": [
        "optimization", "optimisation", "poorly optimized", "performance", "fps", "low fps", "frame drop",
        "stutter", "stuttering", "lag spike", "freeze", "freezing", "crash", "crashes", "crashing",
        "memory leak", "cpu", "gpu", "loading", "load time", "shaders", "unoptimized",
        "оптимизац", "производительн", "фпс", "fps", "просад", "лагает", "лаги", "лагов",
        "фриз", "фризы", "статтер", "вылет", "вылеты", "краш", "крашится", "зависает",
        "долго груз", "загрузка", "утечка памяти", "нагрузка на процессор", "не оптимизирован",
    ],
    "Серверы, пинг и десинхрон": [
        "server", "servers", "ping", "high ping", "latency", "desync", "de-sync", "rubberband",
        "connection", "disconnect", "disconnects", "timeout", "packet loss", "netcode", "matchmaking",
        "queue", "region", "lag compensation", "hitreg", "hit registration",
        "сервер", "серверы", "пинг", "задержк", "десинк", "рассинхрон", "регистрация попаданий",
        "хитрег", "отключ", "дисконнект", "таймаут", "потеря пакетов", "очередь", "регион",
        "матчмейк", "подбор", "неткод", "коннект",
    ],
    "Баланс оружия, брони и матчей": [
        "balance", "balanced", "unbalanced", "overpowered", "op ", "broken weapon", "meta", "nerf",
        "buff", "armor", "armour", "helmet", "damage", "ttk", "time to kill", "one shot", "two shot",
        "recoil", "spread", "match balance", "sweaty", "sweats", "skill gap",
        "баланс", "дисбаланс", "имба", "имбовый", "мета", "нерф", "баф", "броня", "шлем",
        "урон", "ттк", "time to kill", "убивает с одного", "ваншот", "отдача", "разброс",
        "подбор игроков", "скилл", "скиллгеп",
    ],
    "Монетизация и pay-to-win": [
        "pay to win", "pay-to-win", "p2w", "microtransaction", "microtransactions", "mtx", "expensive",
        "overpriced", "cash grab", "paywall", "battle pass", "premium", "bundle", "skin", "skins",
        "loot box", "lootbox", "founder pack", "edition", "dlc", "shop", "monetization", "monetisation",
        "монетизац", "донат", "донатный", "пей ту вин", "пэй ту вин", "p2w", "платн", "дорого",
        "оверпрайс", "магазин", "батл пас", "боевой пропуск", "премиум", "скин", "скины",
        "лутбокс", "пак", "издание", "длс", "заплати", "кошелек",
    ],
    "Прогрессия, гринд, вайпы и лут": [
        "grind", "grindy", "progression", "progress", "wipe", "wipes", "loot", "looting", "farm", "farming",
        "stash", "inventory", "economy", "market", "auction", "craft", "crafting", "quest", "quests",
        "daily", "weekly", "reputation", "rep", "unlock", "gear fear", "extract", "extraction", "risk reward",
        "гринд", "гриндить", "прогресс", "прогрессия", "вайп", "вайпы", "лут", "залутал", "фарм",
        "схрон", "инвентарь", "экономика", "рынок", "аукцион", "крафт", "квест", "задания",
        "ежедневки", "репутация", "разблок", "страх потерять", "экстракт", "эвакуац", "выход",
    ],
    "Стрельба, движение и TTK": [
        "gunplay", "shooting", "shoot", "aim", "aiming", "weapon feel", "recoil", "spread", "movement",
        "movement speed", "sluggish", "clunky", "combat", "hitbox", "hitboxes", "bullet sponge",
        "ttk", "time to kill", "melee", "animation", "inertia", "lean", "peek", "peeker",
        "стрельб", "ганплей", "оружие", "прицел", "аим", "отдача", "разброс", "движение",
        "мувмент", "медленн", "ватн", "неуклюж", "хитбокс", "попадания", "губка для пуль",
        "ттк", "анимац", "инерция", "пик", "выглядыв",
    ],
    "Баги и сломанные механики": [
        "bug", "bugs", "buggy", "glitch", "glitches", "broken", "not working", "doesn't work", "unplayable",
        "softlock", "stuck", "exploit", "abuse", "issue", "issues", "error", "failed", "broken quest",
        "баг", "баги", "багованный", "глюк", "глючит", "сломано", "не работает", "неиграбельн",
        "застрял", "эксплойт", "ошибка", "ошибки", "проблема", "квест сломан", "не засчитало",
    ],
    "Сложность, порог входа и онбординг": [
        "hardcore", "too hard", "difficult", "difficulty", "punishing", "unfair", "new player", "newbie",
        "beginner", "learning curve", "steep learning", "tutorial", "onboarding", "no explanation", "confusing",
        "сложно", "слишком сложно", "хардкор", "несправедливо", "новичк", "новому игроку", "порог вход",
        "кривая обучения", "обучение", "туториал", "онбординг", "ничего не объясняет", "непонятно",
    ],
    "Контент, эндгейм и однообразие": [
        "content", "lack of content", "no content", "endgame", "late game", "boring", "repetitive", "samey",
        "empty", "dead game", "nothing to do", "updates", "patch", "season", "event", "map", "maps", "mode", "modes",
        "контент", "мало контента", "нет контента", "эндгейм", "скучно", "однообраз", "повторя",
        "пусто", "мертвая игра", "нечего делать", "обновлен", "патч", "сезон", "ивент", "карта", "режим",
    ],
    "Интерфейс, UX и лаунчер": [
        "ui", "ux", "interface", "menu", "inventory ui", "stash ui", "launcher", "login", "account",
        "settings", "keybind", "controls", "clutter", "clunky ui", "navigation", "localization", "translation",
        "интерфейс", "меню", "ui", "ux", "инвентарь", "схрон", "лаунчер", "логин", "аккаунт",
        "настройки", "управление", "бинды", "кнопки", "перевод", "локализац", "неудобно",
    ],
    "Звук, визуал и читаемость": [
        "sound", "audio", "footstep", "footsteps", "directional audio", "visual", "graphics", "visibility",
        "dark", "too dark", "blur", "motion blur", "lighting", "fog", "readability", "cluttered", "atmosphere",
        "звук", "аудио", "шаги", "позицион", "график", "графон", "видимость", "темно", "слишком темно",
        "мыло", "блюр", "освещение", "туман", "читаемость", "визуал", "атмосфер",
    ],
    "Комьюнити и токсичность": [
        "toxic", "toxicity", "community", "teamkill", "team kill", "griefer", "griefing", "troll",
        "racist", "abuse", "harassment", "solo", "squad", "randoms", "teammate", "voice chat",
        "токсич", "комьюнити", "сообщество", "тимкилл", "тиммейт", "грифер", "грифинг", "тролль",
        "оскорб", "соло", "сквад", "рандом", "войс", "голосовой чат",
    ],
    "Поддержка, разработчики и коммуникация": [
        "support", "customer support", "dev", "devs", "developer", "developers", "communication", "roadmap",
        "feedback", "ignored", "no response", "refund", "moderation", "promise", "lied", "scam",
        "поддержка", "саппорт", "разработчик", "разрабы", "коммуникац", "роадмап", "обратная связь",
        "игнор", "не отвечают", "возврат", "рефанд", "модерация", "обещали", "обман", "скам",
    ],
}

theme_patterns = {
    name: re.compile("|".join(re.escape(word) for word in words))
    for name, words in THEMES.items()
}

for name, pattern in theme_patterns.items():
    negative_reviews[name] = negative_reviews["text_lower"].str.contains(pattern, na=False)

negative_reviews["количество_найденных_тем"] = negative_reviews[list(THEMES.keys())].sum(axis=1)
negative_reviews.to_csv(output_dir / "negative_reviews_with_themes.csv", index=False, encoding="utf-8")

print("Категорий жалоб размечено:", len(THEMES))
print("Негативных отзывов с хотя бы одной найденной темой:", int((negative_reviews["количество_найденных_тем"] > 0).sum()))
negative_reviews[["game", "language", "review", "количество_найденных_тем"]].head()

In [ ]:
theme_names = list(THEMES.keys())

complaint_theme_summary = pd.DataFrame({
    "категория_жалобы": theme_names,
    "количество_упоминаний_категории": [int(negative_reviews[name].sum()) for name in theme_names],
})

# Один отзыв может попасть сразу в несколько категорий.
# Поэтому эта доля НЕ обязана суммироваться в 100%.
complaint_theme_summary["доля_от_всех_негативных_%"] = (
    complaint_theme_summary["количество_упоминаний_категории"] / max(len(negative_reviews), 1) * 100
).round(1)

# А эта доля показывает структуру всех найденных категорий и суммируется в 100%.
total_theme_mentions = complaint_theme_summary["количество_упоминаний_категории"].sum()
complaint_theme_summary["доля_среди_всех_найденных_категорий_%"] = (
    complaint_theme_summary["количество_упоминаний_категории"] / max(total_theme_mentions, 1) * 100
).round(1)

complaint_theme_summary = complaint_theme_summary.sort_values(
    "количество_упоминаний_категории", ascending=False
).reset_index(drop=True)

complaint_theme_summary.to_csv(output_dir / "negative_theme_summary.csv", index=False, encoding="utf-8")

print(f"Отрицательных отзывов в анализе: {len(negative_reviews)}")
print("Сумма долей по структуре категорий:", complaint_theme_summary["доля_среди_всех_найденных_категорий_%"].sum().round(1), "%")
complaint_theme_summary

## Шаг 9. Каждая игра отдельно

Теперь считаю категории жалоб отдельно для каждой игры. Это важно, потому что у игр одного жанра могут быть разные проблемы:
- у одной главный риск — читы;
- у другой — серверы и десинхрон;
- у третьей — монетизация или нехватка контента.

В таблице ниже проценты считаются внутри негативных отзывов конкретной игры.

In [ ]:
rows = []
for game_name in GAMES:
    game_negative = negative_reviews[negative_reviews["game"] == game_name]
    total = max(len(game_negative), 1)
    row = {"game": game_name, "негативных_отзывов": len(game_negative)}
    for name in theme_names:
        row[name] = round(game_negative[name].sum() / total * 100, 1)
    rows.append(row)

complaints_by_game = pd.DataFrame(rows).set_index("game")
complaints_by_game.to_csv(output_dir / "negative_themes_by_game.csv", encoding="utf-8")

game_points = []
for game_name in GAMES:
    game_row = complaints_by_game.loc[game_name]
    top_categories = game_row[theme_names].sort_values(ascending=False).head(5)
    for place, (category, percent) in enumerate(top_categories.items(), start=1):
        count = int(negative_reviews[(negative_reviews["game"] == game_name) & (negative_reviews[category])].shape[0])
        game_points.append({
            "game": game_name,
            "место": place,
            "категория_жалобы": category,
            "доля_в_негативных_%": percent,
            "количество_отзывов": count,
        })

game_complaint_points = pd.DataFrame(game_points)
game_complaint_points.to_csv(output_dir / "game_complaint_points.csv", index=False, encoding="utf-8")

complaints_by_game

## Шаг 10. Графики по негативным отзывам

Строю графики только по жалобам и сохраняю их в `output/charts/`:
1. Количество негативных отзывов по играм за всё время.
2. Общий топ категорий жалоб.
3. Тепловая карта категорий жалоб по играм.
4. Динамика количества негативных отзывов по месяцам.
5. Динамика топ-категорий жалоб по месяцам.
6. Накопительная динамика негативных отзывов.

Графики помогают увидеть не только «что болит», но и как жалобы менялись во времени.

In [ ]:
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["figure.dpi"] = 110

print("Стиль графиков задан")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_data = complaint_summary.sort_values("негативных_отзывов")
ax.barh(plot_data["game"], plot_data["негативных_отзывов"], color="#d1495b")
for y, v in enumerate(plot_data["негативных_отзывов"]):
    ax.text(v + max(plot_data["негативных_отзывов"]) * 0.01, y, f"{int(v):,}".replace(",", " "), va="center")
ax.set_xlabel("Количество негативных отзывов")
ax.set_title("Сколько негативных отзывов собрано за всё время (ru+en)")
plt.tight_layout()
plt.savefig(charts_dir / "negative_reviews_count_by_game.png", bbox_inches="tight")
plt.show()

In [ ]:
top_complaints = complaint_theme_summary.sort_values(
    "доля_среди_всех_найденных_категорий_%", ascending=True
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    top_complaints["категория_жалобы"],
    top_complaints["доля_среди_всех_найденных_категорий_%"],
    color="#d1495b",
)
for y, v in enumerate(top_complaints["доля_среди_всех_найденных_категорий_%"]):
    ax.text(v + 0.2, y, f"{v}%", va="center")
ax.set_xlabel("Доля среди всех найденных категорий жалоб, %")
ax.set_title("Структура жалоб по категориям (сумма категорий = 100%)")
plt.tight_layout()
plt.savefig(charts_dir / "negative_complaint_categories.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for ax, game_name in zip(axes, GAMES.keys()):
    game_data = game_complaint_points[game_complaint_points["game"] == game_name].sort_values(
        "доля_в_негативных_%", ascending=True
    )
    ax.barh(game_data["категория_жалобы"], game_data["доля_в_негативных_%"], color="#d1495b")
    for y, v in enumerate(game_data["доля_в_негативных_%"]):
        ax.text(v + 0.3, y, f"{v}%", va="center", fontsize=8)
    ax.set_title(game_name)
    ax.set_xlabel("% негативных отзывов игры")

plt.suptitle("Топ-5 жалоб отдельно по каждой игре", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(charts_dir / "top_complaints_by_game.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5.5))
heatmap_data = complaints_by_game[theme_names]
matrix = heatmap_data.values
im = ax.imshow(matrix, cmap="Reds", aspect="auto")

ax.set_xticks(range(len(theme_names)))
ax.set_xticklabels(theme_names, rotation=45, ha="right")
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        value = matrix[i, j]
        color = "white" if value > matrix.max() * 0.6 else "black"
        ax.text(j, i, f"{value:.0f}", ha="center", va="center", color=color, fontsize=8)

ax.set_title("Тепловая карта жалоб: игры × категории (% негативных отзывов игры)")
fig.colorbar(im, ax=ax, label="% негативных отзывов")
plt.tight_layout()
plt.savefig(charts_dir / "negative_complaint_heatmap.png", bbox_inches="tight")
plt.show()

## Шаг 11. Графики с датами

Теперь смотрю динамику по времени:
- сколько негативных отзывов появляется по месяцам у каждой игры;
- какие категории жалоб чаще всего всплывают по месяцам.

Для дат использую `timestamp_created` из Steam. В графиках беру месяц создания отзыва, чтобы не было слишком шумно по дням.

In [ ]:
monthly_negative = negative_reviews.groupby(["month", "game"]).size().reset_index(name="негативных_отзывов")
monthly_negative.to_csv(output_dir / "monthly_negative_reviews.csv", index=False, encoding="utf-8")

monthly_pivot = monthly_negative.pivot(index="month", columns="game", values="негативных_отзывов").fillna(0)

fig, ax = plt.subplots(figsize=(13, 6))
for game_name in monthly_pivot.columns:
    ax.plot(monthly_pivot.index, monthly_pivot[game_name], marker="o", linewidth=2, label=game_name)
ax.set_title("Динамика негативных отзывов по месяцам")
ax.set_xlabel("Месяц")
ax.set_ylabel("Количество негативных отзывов")
ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(charts_dir / "monthly_negative_reviews_by_game.png", bbox_inches="tight")
plt.show()

top_5_categories = complaint_theme_summary.head(5)["категория_жалобы"].tolist()
monthly_theme_rows = []
for month in sorted(negative_reviews["month"].dropna().unique()):
    month_reviews = negative_reviews[negative_reviews["month"] == month]
    for category in top_5_categories:
        monthly_theme_rows.append({
            "month": month,
            "категория_жалобы": category,
            "количество_отзывов": int(month_reviews[category].sum()),
            "доля_месячных_жалоб_%": round(month_reviews[category].sum() / max(len(month_reviews), 1) * 100, 1),
        })

monthly_theme_dynamics = pd.DataFrame(monthly_theme_rows)
monthly_theme_dynamics.to_csv(output_dir / "monthly_top_complaint_categories.csv", index=False, encoding="utf-8")

monthly_theme_pivot = monthly_theme_dynamics.pivot(
    index="month",
    columns="категория_жалобы",
    values="доля_месячных_жалоб_%",
).fillna(0)

fig, ax = plt.subplots(figsize=(13, 6))
for category in monthly_theme_pivot.columns:
    ax.plot(monthly_theme_pivot.index, monthly_theme_pivot[category], marker="o", linewidth=2, label=category)
ax.set_title("Динамика топ-категорий жалоб по месяцам")
ax.set_xlabel("Месяц")
ax.set_ylabel("Доля негативных отзывов месяца, %")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(charts_dir / "monthly_top_complaint_categories.png", bbox_inches="tight")
plt.show()

### Дополнительно: накопительная динамика и тепловая карта по месяцам

Чтобы лучше увидеть изменение отзывов во времени, добавляю ещё два графика:
- накопительное количество негативных отзывов по каждой игре;
- тепловая карта по месяцам и топ-категориям жалоб.

Первый график показывает, когда у игры быстрее накапливался негатив. Второй показывает, какие проблемы становились заметнее в разные месяцы.

In [ ]:
cumulative_pivot = monthly_pivot.sort_index().cumsum()

fig, ax = plt.subplots(figsize=(13, 6))
for game_name in cumulative_pivot.columns:
    ax.plot(cumulative_pivot.index, cumulative_pivot[game_name], marker="o", linewidth=2, label=game_name)
ax.set_title("Накопительное количество негативных отзывов по месяцам")
ax.set_xlabel("Месяц")
ax.set_ylabel("Негативных отзывов накопительно")
ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(charts_dir / "cumulative_negative_reviews_by_game.png", bbox_inches="tight")
plt.show()

category_heatmap = monthly_theme_pivot[top_5_categories].T
fig, ax = plt.subplots(figsize=(13, 5.5))
matrix = category_heatmap.values
im = ax.imshow(matrix, cmap="Reds", aspect="auto")

ax.set_xticks(range(len(category_heatmap.columns)))
ax.set_xticklabels(category_heatmap.columns, rotation=45, ha="right")
ax.set_yticks(range(len(category_heatmap.index)))
ax.set_yticklabels(category_heatmap.index)

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        value = matrix[i, j]
        color = "white" if value > matrix.max() * 0.6 else "black"
        ax.text(j, i, f"{value:.0f}", ha="center", va="center", fontsize=8, color=color)

ax.set_title("Тепловая карта топ-категорий жалоб по месяцам (% негативных отзывов месяца)")
fig.colorbar(im, ax=ax, label="% негативных отзывов месяца")
plt.tight_layout()
plt.savefig(charts_dir / "monthly_complaint_category_heatmap.png", bbox_inches="tight")
plt.show()

## Шаг 12. Итог по негативным отзывам

Собираю короткий текстовый вывод только по негативным отзывам. В нём есть:
- сколько негативных отзывов собрано за всё время;
- какой период покрывают отзывы по каждой игре;
- общий топ проблем жанра;
- отдельные топ-проблемы каждой игры;
- несколько живых примеров негативных отзывов.

Файл сохраняется в `output/negative_analysis_summary.md`.

In [ ]:
lines = []
lines.append("# Выводы по негативным отзывам Steam (extraction-шутеры)\n")
lines.append(f"Собрано негативных отзывов за всё доступное время: {len(negative_reviews)}.")
lines.append(f"Языки: {', '.join(LANGUAGES)}.\n")

lines.append("## Сколько негативных отзывов собрано по играм\n")
for _, row in complaint_summary.iterrows():
    lines.append(
        f"- **{row['game']}** — {int(row['негативных_отзывов'])} негативных отзывов. "
        f"RU: {int(row['russian'])}, EN: {int(row['english'])}. "
        f"Период: {row['первая_дата']} — {row['последняя_дата']}."
    )
lines.append("")

lines.append("## Общий топ категорий жалоб\n")
for _, row in complaint_theme_summary.head(8).iterrows():
    lines.append(
        f"- **{row['категория_жалобы']}** — "
        f"{row['доля_от_всех_негативных_%']}% негативных отзывов содержат эту тему "
        f"({int(row['количество_упоминаний_категории'])} отзывов). "
        f"В структуре всех найденных категорий: {row['доля_среди_всех_найденных_категорий_%']}%."
    )
lines.append("")

lines.append("## Отдельно по каждой игре\n")
for game_name in GAMES:
    lines.append(f"### {game_name}")
    game_top = game_complaint_points[game_complaint_points["game"] == game_name]
    for _, row in game_top.iterrows():
        lines.append(
            f"- {int(row['место'])}. **{row['категория_жалобы']}** — "
            f"{row['доля_в_негативных_%']}% ({int(row['количество_отзывов'])} отзывов)"
        )
    lines.append("")

summary_text = "\n".join(lines)
(output_dir / "negative_analysis_summary.md").write_text(summary_text, encoding="utf-8")

print("Итоговый вывод сохранён в output/negative_analysis_summary.md\n")
print(summary_text[:2000])